# AI Trading Hub — Settimana 0: POC & Validazione Go/No-Go

**Obiettivo**: decidere se Chronos-2 Base ha un edge reale su BTC-PERP 4h prima di investire settimane di sviluppo.

**Criterio go/no-go**: CRPS Chronos-2 < CRPS naive baseline di almeno **5%** su OOS post-cutoff pretrain.

**Sezioni**:
1. Download dati BTC 4h da Hyperliquid REST (24 mesi)
2. Quality check + gap detection
3. Definizione cutoff pretrain Chronos-2 e split IS/OOS
4. Walk-forward CRPS: Chronos-2 vs Naive vs AR(1)
5. SMC baseline: FVG + Liquidity Sweep signal distribution
6. Latency benchmark ciclo completo
7. Decisione Go/No-Go automatica


In [ ]:
import requests
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone, timedelta
from scipy.stats import norm
import torch

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.facecolor"] = "#0f0f0f"
plt.rcParams["figure.facecolor"] = "#1a1a1a"
plt.rcParams["axes.edgecolor"] = "#333"
plt.rcParams["text.color"] = "#e0e0e0"
plt.rcParams["axes.labelcolor"] = "#e0e0e0"
plt.rcParams["xtick.color"] = "#999"
plt.rcParams["ytick.color"] = "#999"
plt.rcParams["grid.color"] = "#222"

# ── Config ───────────────────────────────────────────────────────────────
SYMBOL          = "BTC"
INTERVAL        = "4h"
CONTEXT_LENGTH  = 512    # candele di contesto per Chronos-2
FORECAST_STEPS  = 3      # 3 × 4h = 12h avanti
N_SAMPLES       = 100    # campioni dalla distribuzione Chronos-2
STRIDE_CANDLES  = 30     # un test ogni 30 candele (5 giorni) nel walk-forward
CRPS_GO_THRESHOLD = 0.05 # -5% vs naive = go

# Chronos-2 pretrain cutoff — da verificare nel paper (Sezione 3)
# Chronos-2 pretrain cutoff — Amazon Science 2025 paper, §3 "Data"
# HL BTC-PERP inizia 2024-05-13 → tutta la serie HL è OOS post-cutoff
# Metti 2024-01-01 (mese prima del lancio HL) per avere OOS puro
PRETRAIN_CUTOFF = datetime(2024, 1, 1, tzinfo=timezone.utc)

# Device: MPS su Apple Silicon, CPU su Linux VPS
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"OOS inizia: {PRETRAIN_CUTOFF.strftime('%Y-%m-%d')}")
print(f"Forecast horizon: {FORECAST_STEPS} step x 4h = {FORECAST_STEPS*4}h")


---
## Sezione 1 — Download dati BTC 4h da Hyperliquid REST

Fonte:  endpoint .
I dati storici pubblici non richiedono autenticazione.
Scarichiamo **24 mesi** di candele 4h (≈4380 candele).


In [ ]:
def fetch_hl_candles(
    symbol: str, interval: str,
    start_ms: int, end_ms: int,
    base_url: str = "https://api.hyperliquid.xyz/info"
) -> list[dict]:
    """Fetch candele da Hyperliquid con paginazione automatica."""
    all_candles = []
    chunk_start = start_ms
    # Max ~5000 candele per request — paginiamo a blocchi da 1000
    chunk_ms = 1000 * 4 * 3600 * 1000  # 1000 candele 4h in ms

    while chunk_start < end_ms:
        chunk_end = min(chunk_start + chunk_ms, end_ms)
        payload = {
            "type": "candleSnapshot",
            "req": {
                "coin": symbol,
                "interval": interval,
                "startTime": chunk_start,
                "endTime": chunk_end,
            },
        }
        r = requests.post(base_url, json=payload, timeout=30)
        r.raise_for_status()
        chunk = r.json()
        if not chunk:
            break
        all_candles.extend(chunk)
        chunk_start = chunk[-1]["T"] + 1  # next ms after last close
        time.sleep(0.2)  # rate limit gentile

    # Deduplica per timestamp open
    seen = set()
    deduped = []
    for c in all_candles:
        if c["t"] not in seen:
            seen.add(c["t"])
            deduped.append(c)
    return sorted(deduped, key=lambda x: x["t"])


def candles_to_df(candles: list[dict]) -> pd.DataFrame:
    """Converte lista candele HL in DataFrame con colonne standard."""
    df = pd.DataFrame(candles)
    df["open_time"]  = pd.to_datetime(df["t"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["T"], unit="ms", utc=True)
    for col in ["o", "h", "l", "c", "v"]:
        df[col] = df[col].astype(float)
    df = df.rename(columns={"o": "open", "h": "high", "l": "low",
                             "c": "close", "v": "volume"})
    df = df.set_index("open_time").sort_index()
    return df[["close_time", "open", "high", "low", "close", "volume"]]


In [ ]:
now_ms   = int(datetime.now(timezone.utc).timestamp() * 1000)
start_ms = int((datetime.now(timezone.utc) - timedelta(days=730)).timestamp() * 1000)  # 24 mesi

print(f"Scaricando BTC {INTERVAL} da {datetime.fromtimestamp(start_ms/1000, tz=timezone.utc).date()} ...", end="")
t0 = time.time()
candles_raw = fetch_hl_candles(SYMBOL, INTERVAL, start_ms, now_ms)
df = candles_to_df(candles_raw)
elapsed = time.time() - t0

print(f" fatto in {elapsed:.1f}s")
print(f"Totale candele: {len(df)}")
print(f"Range: {df.index[0].date()} → {df.index[-1].date()}")
print(f"
Prime 3 righe:")
df.head(3)


---
## Sezione 2 — Quality Check

Verifica gap, candele mancanti, outlier di prezzo.


In [ ]:
# Gap detection: atteso 4h tra ogni candela
expected_delta = pd.Timedelta(hours=4)
actual_deltas  = df.index.to_series().diff().dropna()
gaps = actual_deltas[actual_deltas > expected_delta * 1.5]

print("=== Quality Check ===")
print(f"Candele totali:     {len(df)}")
print(f"Attese (730d x 6):  {730 * 6}")
print(f"Gap > 6h trovati:   {len(gaps)}")
if len(gaps) > 0:
    print("Dettaglio gap:")
    for ts, delta in gaps.items():
        print(f"  {ts.date()} — gap di {delta}")

# Outlier: candele con variazione % > 30% in 4h
df["ret_4h"] = df["close"].pct_change()
outliers = df[df["ret_4h"].abs() > 0.30]
print(f"Candele con |ret| > 30%: {len(outliers)}")

# Prezzi NaN
print(f"NaN in close: {df.close.isna().sum()}")

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(df.index, df["close"], color="#00d4ff", linewidth=0.8, label="BTC Close")
ax1.axvline(PRETRAIN_CUTOFF, color="#ff6b35", linewidth=1.5, linestyle="--", label=f"Pretrain cutoff ({PRETRAIN_CUTOFF.date()})")
ax1.set_ylabel("USD")
ax1.set_title("BTC-PERP 4h — Hyperliquid (24 mesi)")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.bar(df.index, df["volume"], color="#444", width=0.1, label="Volume BTC")
ax2.set_ylabel("Volume (BTC)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("poc/btc_overview.png", dpi=120, bbox_inches="tight")
plt.show()
print("
✅ Dati OK — procedi" if len(gaps) < 10 and df.close.isna().sum() == 0 else "
⚠️  Gap rilevanti — investigare prima di procedere")


---
## Sezione 3 — Chronos-2: Cutoff pretrain e split IS/OOS

**IMPORTANTE**: usare Chronos-2 su dati che erano nel suo training set non è un test OOS valido.
Il modello potrebbe avere visto i prezzi BTC durante il pretraining → performance artificialmente gonfiata.

**Cutoff stimato**: ~Gennaio 2024 (da verificare in Section 3.1 del paper Chronos-2).
Fonte: _"Chronos-2: From Univariate to Universal Forecasting"_, Amazon Science, 2025.

> ⚠️ **Da fare manuale**: leggi la sezione "Data" del paper e aggiorna  nella cella Config se diverso da 2024-01-01.


In [ ]:
# Split IS (In-Sample) / OOS (Out-of-Sample)
df_is  = df[df.index <  PRETRAIN_CUTOFF].copy()
df_oos = df[df.index >= PRETRAIN_CUTOFF].copy()

print(f"In-Sample:     {len(df_is)} candele  ({df_is.index[0].date()} → {df_is.index[-1].date()})")
print(f"Out-of-Sample: {len(df_oos)} candele  ({df_oos.index[0].date()} → {df_oos.index[-1].date()})")

if len(df_oos) < 200:
    print("
⚠️  Meno di 200 candele OOS — aggiornare PRETRAIN_CUTOFF nel paper")
else:
    n_test_points = (len(df_oos) - CONTEXT_LENGTH) // STRIDE_CANDLES
    print(f"
Punti di test nel walk-forward: ~{n_test_points}")
    print(f"Durata stimata walk-forward (MPS ~400ms/inference): ~{n_test_points * 0.4:.0f}s")


---
## Sezione 4 — Walk-Forward CRPS: Chronos-2 vs Baselines

**CRPS** (Continuous Ranked Probability Score): misura la qualità di un forecast probabilistico.
Più basso = meglio. Equivale alla MAE per previsioni puntali.

**Setup walk-forward**:
- Ogni  candele nel periodo OOS, prendiamo le ultime  candele come input.
- Forecast: distribuzione Chronos-2 a  (12h avanti).
- Ground truth: prezzo di chiusura reale 12h dopo.
- Confronto: Chronos-2 vs Naive (Gaussiana centrata sul prezzo corrente) vs AR(1).


In [ ]:
from chronos import BaseChronosPipeline

print("Caricamento Chronos-2 Base (~800MB, scarica la prima volta)...")
t0 = time.time()
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
)
print(f"Modello caricato in {time.time()-t0:.1f}s — device: {DEVICE}")


In [ ]:
def crps_from_samples(y_true: float, samples: np.ndarray) -> float:
    """CRPS empirico da campioni (formula energy score). Più basso = meglio."""
    mae    = np.mean(np.abs(samples - y_true))
    spread = np.mean(np.abs(samples[:, None] - samples[None, :])) / 2
    return float(mae - spread)


def crps_gaussian(y_true: float, mu: float, sigma: float) -> float:
    """CRPS per forecast Gaussiano N(mu, sigma)."""
    if sigma <= 0:
        return float(abs(y_true - mu))
    z = (y_true - mu) / sigma
    return float(sigma * (z * (2*norm.cdf(z) - 1) + 2*norm.pdf(z) - 1/np.sqrt(np.pi)))


def ar1_forecast(series: np.ndarray, steps: int) -> float:
    """AR(1) forecast: y_t+1 = phi * y_t + const, proiettato in avanti."""
    if len(series) < 2:
        return series[-1]
    ret = np.diff(series)
    if len(ret) < 2:
        return series[-1]
    phi  = np.corrcoef(ret[:-1], ret[1:])[0, 1]
    phi  = np.clip(phi, -0.99, 0.99)
    pred = series[-1]
    for _ in range(steps):
        pred = pred + phi * (pred - series[-2])
    return float(pred)


In [ ]:
# Walk-forward evaluation
# Utilizziamo solo il periodo OOS (post-cutoff pretrain)
close_oos  = df_oos["close"].values

# Abbiamo bisogno anche di CONTEXT_LENGTH candele IS per i primi punti OOS
close_full = df["close"].values
idx_oos_start = len(df[df.index < PRETRAIN_CUTOFF])

crps_chronos, crps_naive, crps_ar1 = [], [], []
test_timestamps = []
latencies = []

# Punti di test: CONTEXT_LENGTH dopo inizio OOS, ogni STRIDE_CANDLES
test_indices = range(
    idx_oos_start + CONTEXT_LENGTH,
    len(close_full) - FORECAST_STEPS,
    STRIDE_CANDLES
)
total = len(list(test_indices))
print(f"Punti di test: {total}")

for i, idx in enumerate(test_indices):
    context   = close_full[idx - CONTEXT_LENGTH : idx]
    y_true    = close_full[idx + FORECAST_STEPS - 1]  # prezzo reale a 12h
    last_close = context[-1]
    ts = df.index[idx]

    # ── Chronos-2 forecast ───────────────────────────────────────────────
    t_inf = time.time()
    ctx_tensor = torch.tensor(context, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        forecast = pipeline.predict(ctx_tensor, prediction_length=FORECAST_STEPS, num_samples=N_SAMPLES)
    latency_ms = (time.time() - t_inf) * 1000
    latencies.append(latency_ms)

    # samples ha shape (1, N_SAMPLES, FORECAST_STEPS) — prendiamo step finale
    samples_step = forecast[0, :, FORECAST_STEPS - 1].cpu().numpy()
    crps_chronos.append(crps_from_samples(y_true, samples_step))

    # ── Naive baseline: N(last_close, realized_vol) ───────────────────────
    vol_30d = np.std(np.diff(context[-130:])) * np.sqrt(FORECAST_STEPS)
    crps_naive.append(crps_gaussian(y_true, last_close, vol_30d))

    # ── AR(1) baseline ────────────────────────────────────────────────────
    ar1_pred = ar1_forecast(context[-60:], FORECAST_STEPS)
    vol_ar1  = np.std(np.diff(context[-130:])) * np.sqrt(FORECAST_STEPS) * 0.9
    crps_ar1.append(crps_gaussian(y_true, ar1_pred, vol_ar1))

    test_timestamps.append(ts)

    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1:3d}/{total}] {ts.date()} | CRPS: Chronos={crps_chronos[-1]:.1f} Naive={crps_naive[-1]:.1f} AR1={crps_ar1[-1]:.1f} | lat={latency_ms:.0f}ms")

print("
✅ Walk-forward completato")


In [ ]:
# Risultati aggregati
mean_chronos = np.mean(crps_chronos)
mean_naive   = np.mean(crps_naive)
mean_ar1     = np.mean(crps_ar1)
delta_vs_naive = (mean_chronos - mean_naive) / mean_naive  # negativo = meglio
delta_vs_ar1   = (mean_chronos - mean_ar1)   / mean_ar1
mean_latency   = np.mean(latencies)
p95_latency    = np.percentile(latencies, 95)

print("
" + "="*55)
print("       RISULTATI CRPS WALK-FORWARD (OOS)")
print("="*55)
print(f"  Chronos-2 Base:  {mean_chronos:8.2f}")
print(f"  Naive Gaussian:  {mean_naive:8.2f}")
print(f"  AR(1):           {mean_ar1:8.2f}")
print(f"  Delta vs Naive:  {delta_vs_naive:+7.1%}   (target: < -5%)")
print(f"  Delta vs AR(1):  {delta_vs_ar1:+7.1%}")
print("="*55)
print(f"  Latenza media:   {mean_latency:.0f}ms")
print(f"  Latenza p95:     {p95_latency:.0f}ms")
print("="*55)

# Plot CRPS rolling
window = max(5, total // 10)
cr = pd.Series(crps_chronos, index=test_timestamps)
nr = pd.Series(crps_naive,   index=test_timestamps)
ar = pd.Series(crps_ar1,     index=test_timestamps)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
ax1.plot(cr.rolling(window).mean(), color="#00d4ff",  label=f"Chronos-2 (rolling {window})")
ax1.plot(nr.rolling(window).mean(), color="#ff6b35",  label=f"Naive (rolling {window})")
ax1.plot(ar.rolling(window).mean(), color="#a0a0a0",  label=f"AR(1) (rolling {window})")
ax1.set_title("CRPS Walk-Forward OOS — più basso = meglio")
ax1.set_ylabel("CRPS (USD)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(latencies)), latencies, color="#444", label="Latenza inference (ms)")
ax2.axhline(mean_latency, color="#00d4ff", linestyle="--", label=f"Media: {mean_latency:.0f}ms")
ax2.set_title("Latenza Chronos-2 per inference")
ax2.set_ylabel("ms")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("poc/crps_results.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Sezione 5 — SMC Baseline: FVG + Liquidity Sweep

Verifica che le feature SMC producano segnali **non nulli** e **distribuiti uniformemente** nel periodo OOS.
Se i segnali sono tutti concentrati in un mese, la feature è inutile come covariate stazionaria.


In [ ]:
def detect_fvg(df: pd.DataFrame, min_gap_pct: float = 0.001) -> pd.DataFrame:
    """Fair Value Gap: pattern 3-candele con gap di prezzo non coperto."""
    df = df.copy()
    # Bullish FVG: low[i-1] > high[i+1] (gap up)
    df["bullish_fvg"] = (
        (df["low"].shift(-1) > df["high"].shift(1)) &
        ((df["low"].shift(-1) - df["high"].shift(1)).abs() / df["close"] > min_gap_pct)
    )
    # Bearish FVG: high[i-1] < low[i+1] (gap down)
    df["bearish_fvg"] = (
        (df["high"].shift(-1) < df["low"].shift(1)) &
        ((df["high"].shift(-1) - df["low"].shift(1)).abs() / df["close"] > min_gap_pct)
    )
    df["fvg_any"] = df["bullish_fvg"] | df["bearish_fvg"]
    return df


def detect_liquidity_sweep(df: pd.DataFrame, lookback: int = 20) -> pd.DataFrame:
    """Liquidity sweep: prezzo prende swing high/low e richiude nella direzione opposta."""
    df = df.copy()
    swing_high = df["high"].rolling(lookback).max().shift(1)
    swing_low  = df["low"].rolling(lookback).min().shift(1)
    buyside_sweep  = (df["high"] > swing_high) & (df["close"] < swing_high)
    sellside_sweep = (df["low"]  < swing_low)  & (df["close"] > swing_low)
    df["liquidity_sweep"] = buyside_sweep | sellside_sweep
    df["sweep_direction"] = "none"
    df.loc[buyside_sweep,  "sweep_direction"] = "buyside"
    df.loc[sellside_sweep, "sweep_direction"] = "sellside"
    return df


def classify_market_structure(df: pd.DataFrame, lookback: int = 10) -> pd.DataFrame:
    """Market structure: BOS (continuazione) o CHoCH (potenziale inversione)."""
    df = df.copy()
    # Swing highs e lows
    df["swing_high"] = df["high"].rolling(lookback, center=True).max() == df["high"]
    df["swing_low"]  = df["low"].rolling(lookback, center=True).min() == df["low"]
    # Trend corrente: confronta ultimi 2 swing highs e 2 swing lows
    recent_sh = df[df["swing_high"]]["high"].rolling(2).agg(list)
    recent_sl = df[df["swing_low"]]["low"].rolling(2).agg(list)
    df["structure"] = "ranging"
    # BOS up: nuovo HH (Higher High) e HL (Higher Low)
    df["bos_up"]   = df["close"] > df["high"].rolling(lookback).max().shift(1)
    df["bos_down"] = df["close"] < df["low"].rolling(lookback).min().shift(1)
    df.loc[df["bos_up"],   "structure"] = "BOS_up"
    df.loc[df["bos_down"], "structure"] = "BOS_down"
    return df


In [ ]:
# Applica al periodo OOS
df_smc = detect_fvg(df_oos.copy())
df_smc = detect_liquidity_sweep(df_smc)
df_smc = classify_market_structure(df_smc)

# Distribuzione mensile dei segnali
df_smc["month"] = df_smc.index.to_period("M")
monthly = df_smc.groupby("month").agg(
    bullish_fvg=("bullish_fvg", "sum"),
    bearish_fvg=("bearish_fvg", "sum"),
    sweeps=("liquidity_sweep", "sum"),
    bos_up=("bos_up", "sum"),
    bos_down=("bos_down", "sum"),
    total_candles=("close", "count"),
)

print("=== SMC Signal Distribution (OOS) ===")
print(monthly.to_string())

# Rate medie
fvg_rate   = (df_smc["bullish_fvg"].sum() + df_smc["bearish_fvg"].sum()) / len(df_smc)
sweep_rate = df_smc["liquidity_sweep"].sum() / len(df_smc)
print(f"
FVG rate:   {fvg_rate:.1%} delle candele (sano: 5-25%)")
print(f"Sweep rate: {sweep_rate:.1%} delle candele (sano: 2-15%)")

# Plot distribuzione mensile
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
monthly[["bullish_fvg", "bearish_fvg"]].plot(kind="bar", ax=axes[0], color=["#00d4ff", "#ff6b35"], title="FVG Mensili (OOS)")
monthly[["sweeps"]].plot(kind="bar", ax=axes[1], color="#a0ff60", title="Liquidity Sweeps Mensili (OOS)")
monthly[["bos_up", "bos_down"]].plot(kind="bar", ax=axes[2], color=["#00d4ff", "#ff6b35"], title="BOS Mensili (OOS)")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Distribuzione Segnali SMC — OOS Period", y=1.02)
plt.tight_layout()
plt.savefig("poc/smc_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

# Check: segnali distribuiti (non concentrati)
fvg_cv = monthly["bullish_fvg"].std() / (monthly["bullish_fvg"].mean() + 1e-9)
print(f"
FVG CoV (coefficient of variation): {fvg_cv:.2f} (ok se < 2.0)")
if fvg_cv > 2.0:
    print("⚠️  FVG troppo concentrati — feature instabile")
else:
    print("✅ FVG distribuiti correttamente")


---
## Sezione 6 — Latency Benchmark (simulazione ciclo completo)

Target dal piano: **< 3 secondi** per il ciclo completo su VPS.
Su macchina locale con MPS, i numeri saranno migliori — ma danno un lower bound.
Sul VPS Hetzner (CPU, no MPS) la latenza inference sarà 2-3x più alta.


In [ ]:
import ta

def simulate_full_cycle(df_ctx: pd.DataFrame, pipeline, forecast_steps: int = 3) -> dict:
    """Simula il ciclo completo: SMC + indicators + Chronos-2 + decision."""
    t_start = time.perf_counter()

    # 1. SMC features
    t1 = time.perf_counter()
    df_c = detect_fvg(df_ctx)
    df_c = detect_liquidity_sweep(df_c)
    df_c = classify_market_structure(df_c)
    t_smc = (time.perf_counter() - t1) * 1000

    # 2. Technical indicators (ta library)
    t2 = time.perf_counter()
    rsi   = ta.momentum.RSIIndicator(df_c["close"], window=14).rsi().iloc[-1]
    atr   = ta.volatility.AverageTrueRange(df_c["high"], df_c["low"], df_c["close"], window=14).average_true_range().iloc[-1]
    adx   = ta.trend.ADXIndicator(df_c["high"], df_c["low"], df_c["close"], window=14).adx().iloc[-1]
    t_indicators = (time.perf_counter() - t2) * 1000

    # 3. Chronos-2 inference
    t3 = time.perf_counter()
    ctx_tensor = torch.tensor(df_c["close"].values[-512:], dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        fc = pipeline.predict(ctx_tensor, prediction_length=forecast_steps, num_samples=100)
    samples = fc[0, :, -1].cpu().numpy()
    t_chronos = (time.perf_counter() - t3) * 1000

    # 4. Decision logic (placeholder)
    t4 = time.perf_counter()
    current_price    = df_c["close"].iloc[-1]
    directional_prob = float(np.mean(samples > current_price))
    p10 = float(np.percentile(samples, 10))
    p90 = float(np.percentile(samples, 90))
    t_decision = (time.perf_counter() - t4) * 1000

    t_total = (time.perf_counter() - t_start) * 1000
    return {
        "t_smc_ms":        t_smc,
        "t_indicators_ms": t_indicators,
        "t_chronos_ms":    t_chronos,
        "t_decision_ms":   t_decision,
        "t_total_ms":      t_total,
        "directional_prob": directional_prob,
        "p10": p10, "p90": p90,
        "adx": adx, "rsi": rsi, "atr": atr,
    }


# Run 5 cicli per stabilizzare la misura (torch cache)
ctx_sample = df.iloc[-CONTEXT_LENGTH:]
bench_results = []
print("Benchmark ciclo completo (5 run):")
for i in range(5):
    res = simulate_full_cycle(ctx_sample, pipeline, FORECAST_STEPS)
    bench_results.append(res)
    print(f"  Run {i+1}: SMC={res["t_smc_ms"]:.0f}ms | Ind={res["t_indicators_ms"]:.0f}ms | Chronos={res["t_chronos_ms"]:.0f}ms | Tot={res["t_total_ms"]:.0f}ms")

mean_total = np.mean([r["t_total_ms"] for r in bench_results])
print(f"
Media totale (locale, {DEVICE}): {mean_total:.0f}ms")
print(f"Stima VPS Hetzner CX22 (CPU): {mean_total * 3:.0f}ms  (3x conservativo per CPU Intel)")
vps_estimated = mean_total * 3
if vps_estimated < 3000:
    print("✅ Latenza VPS stimata entro target (< 3000ms)")
else:
    print("⚠️  Latenza VPS potrebbe superare 3s — testare sul VPS reale in Settimana 1")


---
## Sezione 7 — Decisione Go/No-Go

Criterio automatico basato sui risultati delle sezioni precedenti.


In [ ]:
print("
" + "="*60)
print("          DECISIONE GO / NO-GO — SETTIMANA 0")
print("="*60)

checks = {}

# 1. CRPS edge vs naive
checks["CRPS edge vs Naive >= 5%"] = delta_vs_naive <= -CRPS_GO_THRESHOLD
print(f"
1. CRPS delta vs Naive:    {delta_vs_naive:+.1%}  " +
      ("✅ GO" if checks["CRPS edge vs Naive >= 5%"] else "❌ NO-GO"))

# 2. CRPS vs AR(1)
checks["CRPS vs AR(1) positive"] = delta_vs_ar1 <= 0
print(f"2. CRPS delta vs AR(1):    {delta_vs_ar1:+.1%}  " +
      ("✅ GO" if checks["CRPS vs AR(1) positive"] else "⚠️  ATTENZIONE"))

# 3. Latenza VPS stimata
checks["Latenza VPS < 3s"] = vps_estimated < 3000
print(f"3. Latenza VPS stimata:    {vps_estimated:.0f}ms  " +
      ("✅ GO" if checks["Latenza VPS < 3s"] else "⚠️  Verificare su VPS reale"))

# 4. SMC signal rate
checks["SMC signals non-null"] = fvg_rate > 0.02 and sweep_rate > 0.01
print(f"4. SMC segnali presenti:   FVG={fvg_rate:.1%} Sweep={sweep_rate:.1%}  " +
      ("✅ GO" if checks["SMC signals non-null"] else "❌ NO-GO"))

# 5. SMC distribuzione
checks["SMC distribuiti"] = fvg_cv < 2.0
print(f"5. SMC distribuzione:      CoV={fvg_cv:.2f}  " +
      ("✅ GO" if checks["SMC distribuiti"] else "⚠️  Concentrati"))

# 6. Dati completi
checks["Dati completi"] = len(df_oos) >= 200 and df.close.isna().sum() == 0
print(f"6. Dati OOS sufficienti:   {len(df_oos)} candele  " +
      ("✅ GO" if checks["Dati completi"] else "❌ NO-GO"))

print("
" + "-"*60)
critical_pass = checks["CRPS edge vs Naive >= 5%"] and checks["SMC signals non-null"] and checks["Dati completi"]
all_pass      = all(checks.values())

if all_pass:
    verdict = "🟢 GO — procedi con Settimana 1"
elif critical_pass:
    verdict = "🟡 GO CONDIZIONALE — procedi ma monitora i warning"
else:
    verdict = "🔴 NO-GO — stop. Analizza CRPS prima di procedere."

print(f"
  VERDETTO FINALE: {verdict}")
print("="*60)

# Salva risultati in JSON per decisions.md
import json
results_export = {
    "date": datetime.now().strftime("%Y-%m-%d"),
    "symbol": SYMBOL,
    "interval": INTERVAL,
    "oos_candles": len(df_oos),
    "pretrain_cutoff": PRETRAIN_CUTOFF.strftime("%Y-%m-%d"),
    "crps_chronos2": round(mean_chronos, 4),
    "crps_naive": round(mean_naive, 4),
    "crps_ar1": round(mean_ar1, 4),
    "delta_vs_naive_pct": round(delta_vs_naive * 100, 2),
    "delta_vs_ar1_pct": round(delta_vs_ar1 * 100, 2),
    "latency_mean_ms": round(mean_latency, 1),
    "latency_p95_ms": round(p95_latency, 1),
    "latency_vps_est_ms": round(vps_estimated, 1),
    "fvg_rate": round(fvg_rate, 4),
    "sweep_rate": round(sweep_rate, 4),
    "fvg_cov": round(fvg_cv, 4),
    "checks": checks,
    "verdict": verdict,
}
with open("poc/week0_results.json", "w") as f:
    json.dump(results_export, f, indent=2)

print(f"
Risultati salvati in poc/week0_results.json")
print("Copia il JSON in decisions.md con la data odierna.")
